# 01 — Meet the plant

## The question

> A treatment plant reports two flow meters. One says **1798 m³/h** of water
> arriving, the other says **1708 m³/h** leaving. **Water is not consumed by a
> treatment plant.** By the end of this notebook you will have a short list of what
> could explain a 5 % gap, and you will know which one to go and check first.

That is the shape of the job. Not *run this query* — a number that cannot be true,
and the work of deciding between three explanations: the instrument is wrong, the
model is wrong, or you are reading the number wrong. Everything in this series is
that move.

**This is notebook 1 of 11, and it is the one you may not need.** It is
orientation: what plant this is, what the tables mean, and the one fact about the
historian that changes every average you will compute afterwards. The analysis here
is deliberately shallow. Notebooks 2 to 11 are where the work is.

| | |
|---|---|
| **01** | *this one* — what is instrumented, and what it costs you to believe it |
| [02](02-three-kinds-of-nothing.ipynb) | no data, bad data, and no change — plus the nine tags that never spoke and the nine that are copies of each other |
| [03](03-choosing-a-tier.ipynb)–[04](04-resampling-buys-you-nothing.ipynb) | raw, 1-minute or 1-hour — and why the rollups are a different estimator, not a faster one |
| [05](05-autocorrelation.ipynb)–[07](07-what-are-you-estimating.ipynb) | how many independent observations you actually have, and what you are averaging over |
| [08](08-rolling-origin-validation.ipynb)–[11](11-dose-or-flow.ipynb) | validating, detecting, dating a change, and blaming one |

**What you need to read this.** Enough SQL to be comfortable with `GROUP BY`, and
enough pandas to read a `DataFrame`. Nothing about time series — that is what the
series is for, and notebook 05 starts from the fact that most people arrive
believing an hour contains 60 independent observations.

**Where the data comes from, because it changes how much you should trust it.**
This week is *synthetic*, generated by a plant model, from a fixed seed. The
chemistry, the flow and the power draw are modelled; the storm is a scheduled
event, and the four instrument faults in the week are injected on purpose. That is
what makes the numbers checkable — every claim in these notebooks is compared
against a real run — and it is also why the findings are about *method* rather than
about this plant. A real plant has its own surprises, which are more interesting
and less tidy.

## Setup

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd

from notebooks._data import connect, storm_window
from notebooks._style import apply_style, save, stamp, trend

apply_style()
pd.set_option("display.width", 130)

conn = connect()

readings = pd.read_sql(
    """
    SELECT signal_id, ts, value, quality
    FROM reading
    WHERE ts >= (SELECT max(ts) FROM reading) - interval '7 days'
    """,
    conn,
)
signals = pd.read_sql("SELECT * FROM signal ORDER BY area, id", conn)
print(f"{len(readings):,} readings across {readings.signal_id.nunique()} signals")

```output
4,239,284 readings across 57 signals
```

## The plant

Wastewater treatment is a sequence of tanks, and the `area` column is that sequence.
Water enters at `INFLUENT`, solids are settled out at `PRIMARY`, the biological
step happens in `AERATION`, the flocs are settled at `SECONDARY`, and what is left
is `EFFLUENT`. `SLUDGE` handles what came out, `UTILITY` is what it costs, `SITE` is
the weather.

Read that before the tables, because it is what makes them legible. A tag called
`SECONDARY:SEC-CL-1:OVERFLOW` is not a mystery: it is the solids leaving a
clarifier, and you can guess what would move it before you query it.

How the 57 tags spread across those areas, and how many of them are machines:

In [ ]:
areas = signals.groupby("area").agg(
    tags=("id", "size"),
)
areas["equipment"] = (
    pd.read_sql("SELECT area, count(*) AS n FROM equipment GROUP BY area", conn)
    .set_index("area")
    .n
)
areas = areas.fillna(0).astype({"equipment": int})
print(areas.sort_values("tags", ascending=False).to_string())

```output
           tags  equipment
area                      
AERATION     14          7
INFLUENT     11          6
EFFLUENT      9          2
SLUDGE        9          3
PRIMARY       5          2
SECONDARY     4          2
SITE          4          0
UTILITY       1          0
```

`SITE` and `UTILITY` have no equipment because they are not machines: the weather
is not a pump, and the power meter is not a blower. `UTILITY:SITE:PLANT_POWER` is
the plant's total electrical draw — the single most useful non-process signal here,
because it is the one thing that responds to *every* process decision at once.


The other tables, and what each is for:

In [ ]:
# **Keyed, not positional.** The first version assigned a plain list:
#
#     counts["what it is"] = ["every change, as it happened", ...]
#
# which lines the descriptions up with the rows *in the order the SQL writes
# them* — and this query ends in `ORDER BY rows DESC`, so it *returns* them in a
# different order. `site` has one row and `event` has none, so the last two came
# back the wrong way round and the table said, in print, that `event` was
# "one row — this is one plant, not many" while showing it at **0**. The
# sentence underneath said the opposite.
#
# Nothing caught it, which is the part worth keeping: every *number* in that
# table was correct, and this project's gate checks numbers. A wrong word in a
# table is invisible to a gate that only reads digits, and a reader who notices
# one stops trusting the other four hundred.
#
# A dict keyed on `table_name` cannot be misaligned by a sort.
counts = pd.read_sql(
    """
    SELECT 'reading' AS table_name, count(*) AS rows FROM reading
    UNION ALL SELECT 'reading_1m', count(*) FROM reading_1m
    UNION ALL SELECT 'reading_1h', count(*) FROM reading_1h
    UNION ALL SELECT 'signal', count(*) FROM signal
    UNION ALL SELECT 'equipment', count(*) FROM equipment
    UNION ALL SELECT 'event', count(*) FROM event
    UNION ALL SELECT 'site', count(*) FROM site
    ORDER BY rows DESC
    """,
    conn,
).set_index("table_name")

counts["what it is"] = pd.Series(
    {
        "reading": "every change, as it happened",
        "reading_1m": "continuous aggregate, 1-minute buckets",
        "reading_1h": "continuous aggregate, 1-hour buckets",
        "signal": "one row per tag: unit, range, deadband, sample rate",
        "equipment": "one row per physical asset, with its failure modes",
        "event": "alarms and state changes — empty in this seed",
        "site": "one row — this is one plant, not many",
    }
)
print(counts.to_string())

```output
               rows                                           what it is
table_name                                                               
reading     4239284                         every change, as it happened
reading_1m   185455               continuous aggregate, 1-minute buckets
reading_1h     5882                 continuous aggregate, 1-hour buckets
signal           57  one row per tag: unit, range, deadband, sample rate
equipment        22   one row per physical asset, with its failure modes
site              1                one row — this is one plant, not many
event             0        alarms and state changes — empty in this seed
```

**Seven tables, 57 tags, 22 pieces of equipment, one week.** The `event` table is
**empty**, and that is not a finding about the plant: alarms are written by the
alarm engine as it consumes the *live* stream, and the seeder writes readings only.
Anything in these notebooks that would like an alarm history has to derive one from
the readings, which is what notebook 09 does.

## A row is not a sample

**This is the one thing to take out of notebook 01, and it is worth reading
before any number below.** Look at what "4,239,284 readings across 57 signals"
does *not* say: over a week, that is 31,872 readings per signal if they were
evenly spread, or one every 1.7 seconds. It is neither. The historian here is
**change-triggered** — it stores a row when the value *differs* — so a tag's row
count is a measure of how much it moved, not how long you watched it.

The spread is enormous, and it starts before the numbers do. Across the week there
are 9,576 signal-hours — 57 tags × 168 hours — and fewer than two thirds of them
wrote a single row:

In [ ]:
counts = (
    readings.set_index("ts")
    .groupby("signal_id")
    .value.resample("1h")
    .count()
    .unstack("signal_id")
)
grid = counts.to_numpy().ravel()
busy = pd.Series(grid[grid > 0])

print(f"signal-hours in the week         : {grid.size:,}")
print(
    f"  wrote something                : {len(busy):,}  "
    f"({100 * len(busy) / grid.size:.1f} %)"
)
print(
    f"  wrote nothing at all           : {grid.size - len(busy):,}  "
    f"({100 * (grid.size - len(busy)) / grid.size:.1f} %)"
)
print()
print("rows in a signal-hour that wrote something:")
print(f"  minimum   {busy.min():,.0f}")
print(f"  1st pct   {busy.quantile(0.01):,.0f}")
print(f"  median    {busy.median():,.0f}")
print(f"  99th pct  {busy.quantile(0.99):,.0f}")
print(f"  maximum   {busy.max():,.0f}")
print(f"  99th pct / 1st pct  {busy.quantile(0.99) / busy.quantile(0.01):,.0f}x")

```output
signal-hours in the week         : 9,576
  wrote something                : 5,881  (61.4 %)
  wrote nothing at all           : 3,695  (38.6 %)

rows in a signal-hour that wrote something:
  minimum   1
  1st pct   1
  median    43
  99th pct  3,600
  maximum   3,600
  99th pct / 1st pct  3,600x
```

**A median signal-hour holds 43 rows. The busiest holds 3,600** — which is a full
hour of a tag declaring a 1,000 ms sample rate, and the ceiling this historian can
produce. So the quietest hours and the loudest differ by more than two orders of
magnitude, and **38.6 % of the grid has no reading in it at all.**

> **An hour is not a sample, and two hours are not twice as much evidence.** Every
> mean you compute is weighted by how much the signal moved while you watched, and
> the plant's most important events are exactly when everything moves. Notebook 04
> is entirely about this; notebook 07 is about deciding which weighting you meant.

This is also why nothing in these notebooks counts rows as evidence of health, and
why `dropna` is treated as a decision rather than a cleanup — notebook 02.

## What the plant does

A weekly hourly mean, because a raw reading count tells you about the historian
and not about the plant. The shaded band is the storm this seed contains, which the
next section is about:

In [ ]:
hourly = (
    readings.set_index("ts")
    .groupby("signal_id")
    .value.resample("1h")
    .mean()
    .unstack("signal_id")
)
storm_start, storm_end = (pd.Timestamp(t) for t in storm_window())

fig, ax = plt.subplots(figsize=(11, 4.5))
trend(ax, {"influent flow": hourly["INFLUENT:FLOW:FLOW"]}, "INFLUENT:FLOW:FLOW")
ax.axvspan(storm_start, storm_end, color="#d62728", alpha=0.18, label="storm")
ax.legend(loc="upper left")
stamp(
    ax,
    hourly["INFLUENT:FLOW:FLOW"],
    window="1 week",
    source="reading (1 h buckets)",
    expected_s=1,
)
ax.set_title("The water arriving — one week, one storm", loc="left")
save(fig, "01_influent")

influent = hourly["INFLUENT:FLOW:FLOW"].mean()
effluent = hourly["EFFLUENT:FLOW:FLOW"].mean()
nh4_in = hourly["INFLUENT:FLOW:NH4_IN"].mean()
nh4_out = hourly["EFFLUENT:FLOW:NH4"].mean()
print(
    f"influent  {influent:.0f} m3/h mean, "
    f"{hourly['INFLUENT:FLOW:FLOW'].min():.0f}-{hourly['INFLUENT:FLOW:FLOW'].max():.0f}"
)
print(f"effluent  {effluent:.0f} m3/h mean")
print(
    f"balance   effluent is {influent - effluent:.0f} m3/h "
    f"({100 * (influent - effluent) / influent:.1f} %) below influent"
)
print(
    f"ammonia   {nh4_in:.1f} mg/L in, {nh4_out:.1f} mg/L out, "
    f"{100 * (1 - nh4_out / nh4_in):.0f} % removed"
)
print(f"nitrate   {hourly['AERATION:AHU-1:NO3_OUT'].mean():.1f} mg/L")
print(f"air       {hourly['AERATION:AHU-1:AIR_FLOW'].mean():.0f} m3/h")
print(
    f"power     {hourly['UTILITY:SITE:PLANT_POWER'].mean():.0f} kW mean, "
    f"peak {hourly['UTILITY:SITE:PLANT_POWER'].max():.0f} kW"
)

```output
influent  1798 m3/h mean, 1265-3996
effluent  1708 m3/h mean
balance   effluent is 90 m3/h (5.0 %) below influent
ammonia   21.8 mg/L in, 7.9 mg/L out, 64 % removed
nitrate   11.2 mg/L
air       6188 m3/h
power     697 kW mean, peak 2032 kW
```

The chemistry is the plant's actual job. Ammonia comes in at **21.8 mg/L** and
leaves at **7.9 mg/L** — **64 % removed** — by bacteria in the aeration basin
converting it to nitrogen gas, which is what `AERATION:AHU-1:NO3_OUT` (**11.2 mg/L**
of nitrate) is the other half of. The blower moves **6188 m³/h** of air to do it,
and the whole plant draws **697 kW** on average, peaking at **2032 kW**.

**Effluent flow is 90 m³/h below influent flow, a 5.0 % deficit.** Water is not
consumed by a treatment plant. So either the two flowmeters do not agree, or the
plant model does not close a mass balance. Either way, do not build a mass balance
on these two numbers without checking which it is.

## Two regimes: a storm, and a daily cycle

Most of the week is one regime and two hours of it are another, and the two need
separating before anything else is measured.

**The storm** first, because it is the loud one. The seed arms a wet-weather event
thirty-six hours before the end of the window:

In [ ]:
flow = hourly["INFLUENT:FLOW:FLOW"].dropna()
lift = hourly["INFLUENT:LIFT:FLOW"].dropna()


def in_storm(s):
    return s[(s.index >= storm_start) & (s.index < storm_end)]


a_day_earlier = lift[
    (lift.index >= storm_start - pd.Timedelta(days=1))
    & (lift.index < storm_end - pd.Timedelta(days=1))
]

tracer = readings[readings.signal_id == "INFLUENT:FLOW:CONDUCTIVITY"]
meter = readings[readings.signal_id == "INFLUENT:FLOW:FLOW"]
meter_in_storm = meter[(meter.ts >= storm_start) & (meter.ts < storm_end)]
nh4 = readings[readings.signal_id == "INFLUENT:FLOW:NH4_IN"]
nh4_storm = nh4[(nh4.ts >= storm_start) & (nh4.ts < storm_end)]

print(f"storm window      {storm_start:%Y-%m-%d %H:%M} to {storm_end:%H:%M} UTC")
print(f"influent flow     {in_storm(flow).max():.0f} m3/h at the peak hour")
print(f"same hour, day before, lift station: {a_day_earlier.iloc[0]:.0f} m3/h")
print(
    f"flow meter        {len(meter_in_storm):,} of {len(meter):,} readings "
    f"({100 * len(meter_in_storm) / len(meter):.1f} %) fall in "
    f"{100 * 2 / 168:.1f} % of the week"
)
print(
    f"influent NH4      {nh4_storm.value.mean():.1f} mg/L in the storm, "
    f"{nh4[~nh4.index.isin(nh4_storm.index)].value.mean():.1f} outside it"
)
print(
    f"conductivity      {len(tracer)} readings all week, "
    f"{int(((tracer.ts >= storm_start) & (tracer.ts < storm_end)).sum())} in the storm"
)

```output
storm window      2026-09-27 12:00 to 14:00 UTC
influent flow     3996 m3/h at the peak hour
same hour, day before, lift station: 2012 m3/h
flow meter        1,162 of 3,733 readings (31.1 %) fall in 1.2 % of the week
influent NH4      14.1 mg/L in the storm, 22.4 outside it
conductivity      357 readings all week, 355 in the storm
```

**Influent flow doubles at the peak hour: 3996 m³/h against 2012 the day
before.** And the storm *dilutes* the influent — ammonia falls to **14.1 mg/L** against **22.4** outside it —
so the plant's hydraulic load spikes at the same moment its pollutant load per litre
drops. That is the whole reason a storm is hard to see with one threshold, and hard
to average: two things a naive analysis treats as one move in opposite directions.

The line to hold on to is the third. **31.1 % of the flow meter's readings come from
1.2 % of the week.** The historian stores a row when the value changes, so a signal
that moves constantly for two hours writes almost a third of its week in them. That
is not a curiosity about this meter; it is the mechanism behind every disagreement
between averages later in this notebook, and notebook 04 is about it.

`INFLUENT:FLOW:CONDUCTIVITY` is the scenario's declared cheap tracer, and it earns
the description: **357** readings across the week, **355** of them inside the storm. A
signal that says almost nothing for six days and speaks for two hours is exactly what
change-triggered storage is for, and exactly what `dropna` and `ffill` cannot
distinguish from a dead sensor — notebook 02.

**The daily cycle** is the other regime, and it is only visible once the storm is
out of the way. Removing it, and the three hours after it while the plant settles:

In [ ]:
calm = flow[
    (flow.index < storm_start) | (flow.index >= storm_end + pd.Timedelta(hours=3))
]
profile = calm.groupby(calm.index.hour).median()

print(f"hours of the week with a flow reading : {len(flow)} of 168")
print(f"hours left once the storm is removed  : {len(calm)}")
print(f"peak   {profile.idxmax():02d}:00 UTC  {profile.max():.0f} m3/h")
print(f"trough {profile.idxmin():02d}:00 UTC  {profile.min():.0f} m3/h")
print(f"ratio  {profile.max() / profile.min():.2f}x")

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.bar(profile.index, profile.values, color="#1f77b4", width=0.85)
ax.set_xticks(range(0, 24))
ax.set_xlabel("hour of day (UTC)")
ax.set_ylabel("influent flow  [m3/h]")
ax.set_title("One day of influent flow — median by hour, storm removed", loc="left")
stamp(
    ax,
    calm,
    window="1 week less the storm",
    source="reading (1 h buckets)",
    expected_s=1,
    extra="hours 11-14 have no reading: the flow was steady",
)
save(fig, "01_diurnal")

```output
hours of the week with a flow reading : 143 of 168
hours left once the storm is removed  : 138
peak   07:00 UTC  2206 m3/h
trough 00:00 UTC  1265 m3/h
ratio  1.74x
```

**A 1.74× swing, peaking at 07:00 and bottoming at 00:00.** Four hours of the day
(11:00 to 14:00) have no bar at all, and that is not a plotting defect: the flow was
inside its deadband for those hours on every calm day, so the historian wrote
nothing. Notebook 02 is the difference between that and a dead meter.

Read the *hour* with suspicion. The seeder starts the cycle when the window starts,
and this window starts at midnight, so "peaks at 07:00" describes the generator's
convention and not a catchment. An earlier seed of this same repository, started
fifteen hours later in the day, peaked fifteen hours later — the shape is the
plant's and the phase is the seeder's. Report the amplitude, and be wary of anyone
who reports the hour.

Hold on to the *shape*. It is the reason notebooks 05 through 11 are hard: one
shared daily cycle drives nearly every signal in the plant, so a model that has
learned the calendar will score well and predict nothing.

## Takeaways

Four things, and the third is the one that will still be true in notebook 11.

- **Seven tables, 57 tags, 22 assets, one week.** `signal` is the one to read first:
  it holds `unit`, `normal_low`/`high`, `deadband` and `sample_ms` for every tag,
  and every one of those is a thing you would otherwise hard-code. `area` is the
  process — `INFLUENT → PRIMARY → AERATION → SECONDARY → EFFLUENT` — so a tag name
  is a prediction before you query it.
- **The plant's job is chemistry, and you can see it in three numbers.** Ammonia
  arrives at **21.8 mg/L** and leaves at **7.9 mg/L**, **64 %** removed, by bacteria
  spending **6188 m³/h** of air and drawing **697 kW**.
- **A row is not a sample.** This historian is change-triggered, so a median
  signal-hour holds **43** readings, the busiest **3,600**, and **38.6 %** of the
  grid has none at all. Every mean you compute is weighted by how much the signal
  moved while you watched — and the plant's most important events are exactly when
  everything moves.
- **Two regimes, and they need separating before anything is measured.** The storm
  is two hours carrying **31.1 %** of the flow meter's week; the daily cycle swings
  **1.74×**. And the hour the cycle peaks in belongs to the seeder, not the plant —
  report the amplitude, and be suspicious of anyone who reports the hour.

**The mass balance does not close, and this notebook does not pretend to know
why.** Influent averages **1798 m³/h** and effluent **1708 m³/h**, a 5.0 % deficit
that water treatment does not explain. Either the two meters do not agree or the
plant model does not close a balance, and this dataset cannot tell you which.
Exercise 4 is the way to find out, and it is the honest end of an orientation
notebook: a number you can compute, and a reason you cannot yet defend.

## Exercises

1. **Ask whether the mass balance closes.** Compare total volume in and out over
   the week using `INFLUENT:FLOW:FLOW` and `EFFLUENT:FLOW:FLOW`, flow-weighted
   rather than time-averaged. Does the 5 % deficit survive? Should it?
2. **Find the cycle's period.** The signal is daily. Prove it: compute the
   autocorrelation of `INFLUENT:FLOW:FLOW` out to 72 hours and show the peak. Then
   explain why the same plot for a signal that writes 3,600 rows an hour looks
   nothing like it. *Notebook 05.*
3. **Rank the tags by how much they move.** Using the per-signal-hour counts above,
   which five tags are quietest, and what does that say about which ones you would
   trust to answer a question without checking? *Notebook 02.*
4. **The tag list is not the measurement list.** `signal` has 57 rows and
   `equipment` has 22, but only some tags measure something a person would ask
   about. Propose a rule that separates them, and say which column you would use.
   *Notebook 02, on duplicates.*

---

**Next:** [02 — Three kinds of nothing](02-three-kinds-of-nothing.ipynb) ·
**Back to the series** [README](README.md)